# 2. Clusters thématiques (Louvain)

91 % des renvois restent dans un même acte : sur les renvois seuls, Louvain retrouve les actes. On compare ce graphe « structure » au graphe « thèmes » (renvois + 10 plus proches voisins sémantiques), retenu pour le livrable. Les clusters sont recalculés ici avec le code de `graphrag/clusters.py`, sans rien écrire dans Neo4j.

_Notebook rejoué avec le code du projet (`graphrag/`, et `app/` du projet vectoriel). Ses sorties sont réelles : il a été exécuté le 10/11/2026 à 03:04 par `scripts/build_notebooks.py`. Aucun appel au LLM n'est fait ici ; Neo4j doit être démarré._

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# candidats de recherche déjà calculés (MultiQuery, HyDE) : aucune requête au LLM
os.environ.setdefault("SEED_CACHE_PATH", str(ROOT / "data" / "cache" / "user_test_candidates.json"))

import pandas as pd
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)

from graphrag import config
config.use_vector_rag()
DATA = config.VECTOR_RAG_DIR / "data" / "eurlex"
RESULTS = ROOT / "results"
print("Projet :", ROOT)
print("Données du projet vectoriel :", DATA)

Projet : C:\Users\hazem\5eme\GraphRAG-juridique
Données du projet vectoriel : C:\Users\hazem\5eme\Advenced-Rag-Agents-main\Advenced-Rag-Agents-main\data\eurlex


In [2]:
from graphrag import clusters as cl

ids, celex, titles, emb, refs, nexts, subjects = cl.load()
internes = sum(celex[a] == celex[b] for a, b in refs)
print(len(ids), "articles ;", len(refs), "renvois, dont", f"{internes / len(refs):.0%}", "à l'intérieur d'un acte")

3941 articles ; 7047 renvois, dont 91% à l'intérieur d'un acte


In [3]:
structure = {tuple(sorted(e)): 1.0 for e in refs + nexts}
themes = cl.knn_edges(ids, emb)                       # 10 plus proches voisins, poids = cosinus
for a, b in refs:
    key = tuple(sorted((a, b)))
    themes[key] = themes.get(key, 0.0) + cl.REFERS_WEIGHT

comm_s, mod_s = cl.louvain(ids, structure)
comm_t, mod_t = cl.louvain(ids, themes)
pd.DataFrame({
    "structure (REFERS_TO + NEXT)": {"modularite": round(mod_s, 3), **cl.describe(comm_s, celex, ids)},
    "themes (REFERS_TO + 10 voisins), retenu": {"modularite": round(mod_t, 3), **cl.describe(comm_t, celex, ids)},
})

,structure (REFERS_TO + NEXT),"themes (REFERS_TO + 10 voisins), retenu"
modularite,0.917,0.814
n_clusters,57.000,25.000
n_singletons,0.000,0.000
taille_max,284.000,457.000
taille_mediane,44.000,139.000
nmi_avec_acte,0.920,0.738
clusters_de_3_articles_ou_plus,54.000,25.000
clusters_de_3_articles_ou_plus_couvrant_plusieurs_actes,23.000,19.000
part_articles_dans_cluster_multi_actes,0.581,0.857


## Les 25 clusters retenus

Étiquette : descripteurs EuroVoc sur-représentés (couverture × log(1 + lift)), ou « Clauses formelles » si au moins 40 % des titres d'articles sont des clauses types.

In [4]:
report = json.loads((RESULTS / "clusters.json").read_text(encoding="utf-8"))
print("Recalcul identique au fichier enregistré :",
      report["themes"]["n_clusters"] == len(comm_t) and report["themes"]["modularite"] == round(mod_t, 3))
pd.DataFrame(report["clusters"])[["id", "size", "n_acts", "label"]]

Recalcul identique au fichier enregistré : True


,id,size,n_acts,label
0,0,457,20,protection de la vie privée · données personnelles · protection des données
1,1,359,17,aide financière · budget général (UE) · contrat public
2,2,276,1,exportation (UE) · réglementation douanière · territoire douanier (UE)
3,3,231,33,contrat · rapprochement des législations · droit d'auteur
4,4,229,10,Frontex · migration de retour · contrôle à la frontière
5,5,228,6,"biométrie · droit d'asile · espace de liberté, sécurité et justice"
6,6,213,24,politique des consommateurs · échange d'information · violation du droit de l'UE
7,7,190,7,surveillance du marché · sécurité du produit · autorisation de vente
8,8,186,6,prestation de services · marché unique numérique · diffusion de l'information
9,9,176,6,sécurité des systèmes d’information · criminalité informatique · Agence de l'Union européenne pour la cybersécurité


In [5]:
from graphrag.db import run

pd.DataFrame(run("""MATCH (c:Cluster) OPTIONAL MATCH (a:Article)-[:IN_CLUSTER]->(c)
    RETURN count(DISTINCT c) AS clusters, count(a) AS articles_rattaches""")).assign(
    articles_sans_cluster=run("MATCH (a:Article) WHERE NOT (a)-[:IN_CLUSTER]->() RETURN count(a) AS n")[0]["n"])

,clusters,articles_rattaches,articles_sans_cluster
0,25,3941,0


In [6]:
# Cluster « intelligence artificielle » : ses renvois internes (requête utilisée pour la capture Neo4j)
pd.DataFrame(run("""MATCH (c:Cluster {id: 10})<-[:IN_CLUSTER]-(a:Article)-[r:REFERS_TO]->(b:Article)-[:IN_CLUSTER]->(c)
    RETURN c.label AS cluster, count(r) AS renvois_internes, count(DISTINCT a.celex) AS actes"""))

,cluster,renvois_internes,actes
0,nouvelle technologie · technologie intelligente · intelligence artificielle,305,5
